# Week 6: Pair Programming - Network Architects (Visual Edition)

**Exercise:** See how architecture changes the decision boundary  
**Time:** 15 minutes  
**Scaffolding:** 60% provided (guided experimentation)  
**Partners:** Work in pairs!

---

## 🎯 Your Mission

Last time you couldn't *see* what the network was doing to MNIST (784 dimensions — no way to draw it). Today the data is **2-D**, so every experiment prints a **picture of the decision boundary** plus the **parameter count**.

**Your tasks:**
1. Run the baseline model (provided)
2. Experiment 1: **Add** hidden layers (go deep & wide)
3. Experiment 2: **Remove** the hidden layer (go linear)
4. Experiment 3: **Change the width** (go narrow)
5. Compare all the boundaries side by side!

**Key:** **Predict → Run → Discuss** with your partner. Before you run each cell, *sketch* what you think the boundary will look like.

---

## Setup (Provided)

Run this once. It loads the data and defines two helpers you'll reuse: `plot_boundary()` and `run()`.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# A SMALL, noisy training set -> architecture choices show up clearly.
# A big separate test set -> a stable accuracy estimate.
X_train, y_train = make_moons(n_samples=40,   noise=0.28, random_state=1)
X_test,  y_test  = make_moons(n_samples=3000, noise=0.28, random_state=99)

# A grid of points covering the plane, reused for every boundary plot.
_all = np.vstack([X_train, X_test]); _pad = 0.5
xx, yy = np.meshgrid(
    np.linspace(_all[:,0].min()-_pad, _all[:,0].max()+_pad, 300),
    np.linspace(_all[:,1].min()-_pad, _all[:,1].max()+_pad, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

# We'll stash each trained model + its scores here for the final comparison.
models, results = {}, {}

def plot_boundary(model, title, ax=None):
    """Colour the plane by the model's prediction, then drop the training points on top."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    Z = (model.predict(grid, verbose=0) > 0.5).astype(int).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.30, cmap='coolwarm')
    ax.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap='coolwarm',
               s=55, edgecolors='k', linewidths=0.7)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    return ax

def run(model, name):
    """Train, score on train+test, count params, plot the boundary, and remember it."""
    model.fit(X_train, y_train, epochs=500, verbose=0)
    train_acc = model.evaluate(X_train, y_train, verbose=0)[1]
    test_acc  = model.evaluate(X_test,  y_test,  verbose=0)[1]
    n_params  = model.count_params()
    models[name]  = model
    results[name] = {'train': train_acc, 'test': test_acc, 'params': n_params}
    plot_boundary(model, f"{name}\ntrain {train_acc:.2f} / test {test_acc:.2f}  |  {n_params:,} params")
    plt.show()
    print(f"{name}: train {train_acc:.2f} | test {test_acc:.2f} | {n_params:,} params")

print("✓ Data loaded, helpers ready")

---

## Baseline Model (Run This First!)

One hidden layer, 32 neurons. This is our reference point.

In [ ]:
# Baseline: 1 hidden layer (32 neurons)
baseline_model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1,  activation='sigmoid')   # 1 output neuron for 2-class problem
])

baseline_model.compile(optimizer='adam',
                       loss='binary_crossentropy',
                       metrics=['accuracy'])

baseline_model.summary()
run(baseline_model, 'Baseline (1x32)')

**Look at the boundary:** it should be a smooth curve that follows the two moons. This is a network with *enough* capacity for this problem.

**Now let's experiment!**

---

## Experiment 1: Add Hidden Layers (Go Deep & Wide)

**Predict with your partner:** If we give the network *way* more capacity — 3 hidden layers of 256 neurons — will the test accuracy go up? What will the boundary look like?

**TODO:** Fill in the three big hidden layers.

In [ ]:
# TODO: build a MUCH bigger network -> 3 hidden layers, 256 neurons each, ReLU.
#       (Want to add even more? Just paste another Dense line. Want fewer? Delete one.)

experiment1_model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(____, activation='____'),
    layers.Dense(____, activation='____'),
    layers.Dense(____, activation='____'),
    layers.Dense(1, activation='sigmoid')
])

experiment1_model.compile(optimizer='adam',
                          loss='binary_crossentropy',
                          metrics=['accuracy'])

experiment1_model.summary()
run(experiment1_model, 'Deep & wide (3x256)')

**Discuss with partner:**
- Compare **train** vs **test** accuracy. Which one shot up? (Hint: check how close train is to 1.00.)
- Look at the boundary — is it a clean curve, or does it wiggle around individual dots?
- How many parameters does this model have vs the baseline? Was the extra capacity *worth* it?

---

## Experiment 2: Remove the Hidden Layer (Go Linear)

**Predict:** What happens with **zero** hidden layers — just input straight to output? What shape *must* the boundary be?

**TODO:** Notice there's no hidden layer here at all. Run it and see what a model with no hidden layer can (and can't) do. Then, if you have time, try adding **one** small hidden layer back (e.g. `Dense(4)`) and watch the boundary start to bend.

In [ ]:
# No hidden layers -- this is just input -> output (a linear classifier).
experiment2_model = keras.Sequential([
    keras.Input(shape=(2,)),
    # (no Dense hidden layer here on purpose!)
    layers.Dense(1, activation='sigmoid')
])

experiment2_model.compile(optimizer='adam',
                          loss='binary_crossentropy',
                          metrics=['accuracy'])

experiment2_model.summary()
run(experiment2_model, 'Linear (0 hidden)')

**Discuss:**
- The boundary is a straight line. Why *can't* it be anything else with no hidden layer?
- How does it do on the curvy moons compared to the baseline?
- Is a simpler model always worse? When might a straight line be exactly what you want?

---

## Experiment 3: Change the Width (Go Narrow)

**Predict:** Keep it to one hidden layer, but shrink it to just **4 neurons**. Enough to bend around the moons, or not quite?

**TODO:** Fill in one narrow hidden layer (4 neurons).

In [ ]:
# TODO: ONE hidden layer, but NARROW -> 4 neurons, ReLU.
#       (Try changing 4 to 8, 16, 64 later and watch the boundary sharpen.)

experiment3_model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(____, activation='____'),
    layers.Dense(1, activation='sigmoid')
])

experiment3_model.compile(optimizer='adam',
                          loss='binary_crossentropy',
                          metrics=['accuracy'])

experiment3_model.summary()
run(experiment3_model, 'Narrow (1x4)')

**Discuss:**
- Compare the narrow (4) boundary to the baseline (32). What did the extra neurons buy you?
- Does *doubling* neurons double accuracy? Where do the returns start shrinking?

---

## Results Summary

In [ ]:
import pandas as pd

order = ['Baseline (1x32)', 'Deep & wide (3x256)', 'Linear (0 hidden)', 'Narrow (1x4)']
rows = []
for name in order:
    if name in results:
        r = results[name]
        rows.append({'Model': name,
                     'Params': f"{r['params']:,}",
                     'Train acc': f"{r['train']:.2f}",
                     'Test acc': f"{r['test']:.2f}",
                     'Gap (train-test)': f"{r['train']-r['test']:+.2f}"})

print(pd.DataFrame(rows).to_string(index=False))

## All Four Boundaries, Side by Side

Ordered by capacity: **fewest parameters → most**. Read left to right and watch the boundary go from a flat line, to a clean curve, to a wiggly overfit.

In [ ]:
# Show every boundary we trained, in order of increasing capacity.
grid_order = ['Linear (0 hidden)', 'Narrow (1x4)', 'Baseline (1x32)', 'Deep & wide (3x256)']
grid_order = [n for n in grid_order if n in models]   # only the ones you've run

fig, axes = plt.subplots(1, len(grid_order), figsize=(5*len(grid_order), 5))
if len(grid_order) == 1:
    axes = [axes]
for ax, name in zip(axes, grid_order):
    r = results[name]
    plot_boundary(models[name],
                  f"{name}\ntrain {r['train']:.2f} / test {r['test']:.2f}\n{r['params']:,} params",
                  ax=ax)
plt.suptitle("Same data, four architectures", fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

---

## 🎓 Key Insights

**Fill in with your partner:**

1. Removing the hidden layer forced the boundary to be a ____________, which on the moons was ____________.

2. Adding lots of capacity (3×256) sent **train** accuracy to about ______ but **test** accuracy ____________, and the boundary got ____________.

3. Going from 4 neurons to 32 neurons changed the boundary by ____________________________.

4. The gap between train and test accuracy tells us ____________________________.

**Biggest surprise:** ___________________________________________________

---

## 🚀 Bonus (If Time)

- **Break the overfit:** shrink `n_samples` in the training set from 40 to 20. Does the big model overfit *even harder*?
- **Fix the overfit:** add `layers.Dropout(0.3)` between the big Dense layers in Experiment 1. What happens to the gap?
- **Swap the activation:** change `'relu'` to `'tanh'` in the hidden layers. Better or worse?

---

*These patterns — too little capacity underfits, too much overfits — transfer to ANY neural network problem, including the MNIST one from before.*